In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import psycopg2
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
def loc_pings(l):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        #print(url)
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        print("LOC PINGS A GAYI")
        #print(".")

        
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:
loc_pings([100001, 1583032576, 1583032596])

In [ ]:
gps_vehicles=pd.read_sql("select vehicle_id from vehicle_details",engine)['vehicle_id']
gps_vehicles=[i for i in gps_vehicles if i>0]

In [ ]:
vehicle_fastag=pd.read_sql("select vehicle_id,fastag_id,fastag_class,vehicle_number from analytics.vehicle_tags",con) 
vehicle_fastag=vehicle_fastag[vehicle_fastag['vehicle_id'].isin(gps_vehicles)]
vehicle_fastag_list=[i for i in vehicle_fastag['fastag_id'] if i>0]  



In [ ]:
vehicle_fastag

In [ ]:
len(vehicle_fastag_list)

In [ ]:
existing_plazas=pd.read_sql("select plaza_code from plaza_code_lat_lons",engine)['plaza_code'].tolist()
existing_plazas=[i for i in existing_plazas if i!=None]

In [ ]:
total_plazas=pd.read_sql("select distinct plazacode from fleet_lake.fastagtransaction ",con)          


In [ ]:
remaining_plazas=total_plazas[~total_plazas['plazacode'].isin(existing_plazas)]['plazacode'].tolist()
remaining_plazas=[i for i in remaining_plazas if i!=None] 


In [ ]:
len(remaining_plazas)

In [ ]:
len(total_plazas)

In [ ]:
fastag_transactions_total=pd.DataFrame()
for i in range(0,75000,1000):
    fastag_transactions=pd.read_sql("select plazacode,fastagid,amount,time from fleet_lake.fastagtransaction where transactiontype='DEBIT' and amount>0 and fastagid in {} and plazacode in {}".format(tuple(vehicle_fastag_list[i:i+1000]),tuple(remaining_plazas)),con)   
    fastag_transactions_total=pd.concat([fastag_transactions,fastag_transactions_total]) 
    print(i)
    print(len(fastag_transactions))
    
    

In [ ]:
vehicle_time_plaza=pd.merge(fastag_transactions_total,vehicle_fastag,left_on='fastagid',right_on='fastag_id',how='left')  
vehicle_time_plaza['start_time']=vehicle_time_plaza['time']-10
vehicle_time_plaza['end_time']=vehicle_time_plaza['time']+10
vehicle_time_plaza=vehicle_time_plaza.dropna()
vehicle_time_plaza['start_time']=vehicle_time_plaza['start_time'].apply(int)
vehicle_time_plaza['end_time']=vehicle_time_plaza['end_time'].apply(int)
vehicle_time_plaza['vehicle_id']=vehicle_time_plaza['vehicle_id'].apply(int)




In [ ]:
def plaza_centroid(plazacode):
    try:
        vehicle_time_plaza_reuse=vehicle_time_plaza[vehicle_time_plaza['plazacode']==str(plazacode)]
        data=vehicle_time_plaza_reuse[['vehicle_id','start_time','end_time']].values.tolist()
        print(data)
        import multiprocessing
        pool = multiprocessing.Pool()
        results=pool.map(loc_pings,data[0:200])
        pool.close()
        pool.join()
        data_lat_longs=pd.concat(results)
        #print(data_lat_longs)
        print("data pings aa gaye")
        data_lat_longs
        data_lat_longs[['longitude','latitude']] = pd.DataFrame(data_lat_longs.location.values.tolist(), index= data_lat_longs.index)
        data_lat_longs['latitude']=data_lat_longs['latitude'].apply(lambda x : round(x,3))
        data_lat_longs['longitude']=data_lat_longs['longitude'].apply(lambda x : round(x,3))
        data_lat_longs['loc_str'] = data_lat_longs.apply(lambda row: str(int(row['latitude']*1000))+str(int(row['longitude']*1000)), axis=1)
        data_lat_longs=data_lat_longs[data_lat_longs['loc_str']==data_lat_longs['loc_str'].value_counts().idxmax()]
        import numpy as np
        from sklearn.cluster import KMeans
        X=np.array(data_lat_longs['location'].tolist())
        print("kmeans start ho gaya")
        kmeans = KMeans(n_clusters=1, init='k-means++', n_init=10)
        pred_y = kmeans.fit_predict(X)
        plaza_lat_long=kmeans.cluster_centers_[0]
        print("kmeans end ho gaya")

        return [plazacode,plaza_lat_long]
    except Exception as e:
        print(e)
        return None


In [ ]:
import multiprocessing
pool = multiprocessing.pool.Pool()
yolo=pool.map(plaza_centroid,vehicle_time_plaza['plazacode'].unique()[0:10])
pool.close()
pool.join()


In [ ]:
vehicle_fastag=vehicle_fastag[vehicle_fastag['vehicle_id'].isin(gps_vehicles)]
vehicle_fastag_list=[i for i in vehicle_fastag['fastag_id'] if i>0]
#print(vehicle_fastag_list)

In [ ]:
        plazacode='378001'
        vehicle_time_plaza_reuse=vehicle_time_plaza[vehicle_time_plaza['plazacode']==str(plazacode)]
        data=vehicle_time_plaza_reuse[['vehicle_id','start_time','end_time']].values.tolist()
        import multiprocessing
        pool = multiprocessing.Pool(30)
        results=pool.map(loc_pings,data[0:1000])
        pool.close()
        pool.join()
        data_lat_longs=pd.concat(results)
        
        data_lat_longs[['longitude','latitude']] = pd.DataFrame(data_lat_longs.location.values.tolist(), index= data_lat_longs.index)
        data_lat_longs['latitude']=data_lat_longs['latitude'].apply(lambda x : round(x,3))
        data_lat_longs['longitude']=data_lat_longs['longitude'].apply(lambda x : round(x,3))
        data_lat_longs['loc_str'] = data_lat_longs.apply(lambda row: str(int(row['latitude']*1000))+str(int(row['longitude']*1000)), axis=1)
        data_lat_longs=data_lat_longs[data_lat_longs['loc_str']==data_lat_longs['loc_str'].value_counts().idxmax()]
 
        #print(data_lat_longs)
        print("data pings aa gaye")
        import numpy as np
        from sklearn.cluster import KMeans
        X=np.array(data_lat_longs['location'].tolist())
        print("kmeans start ho gaya")
        kmeans = KMeans(n_clusters=1, init='k-means++', n_init=20)
        pred_y = kmeans.fit_predict(X)
        plaza_lat_long=kmeans.cluster_centers_[0]
        print("kmeans end ho gaya")
        plt.scatter(X[:, 0], X[:, 1], s=1, cmap='viridis')

        centers = kmeans.cluster_centers_
        plt.scatter(centers[:, 0], centers[:, 1], c='yellow', s=30, alpha=1);
        plt.autoscale(enable=False)

In [ ]:
yolo=[]
for i in vehicle_time_plaza['plazacode'].unique():
    print(i)
    a=plaza_centroid(i)
    yolo.append(a)
    print(a)


In [ ]:
vehicle_time_plaza['plazacode'].nunique()

In [ ]:
yolo

In [ ]:
plaza_centroid('378001')

In [ ]:
final=[]
for i in yolo:
    yo=[i[0],i[1][0],i[1][1]]
    final.append(yo)

In [ ]:
plaza_lats_lons=pd.DataFrame(final,columns=['plaza_code','lat','lon'])